# Notebook 01 - OULAD join and weekly-window, cutoff-aware features

Builds the feature matrix for the Trustworthy Explainable Early-Warning project.

**Discipline.** Every feature summarises only activity observed up to a fixed week cutoff
(weeks 5, 10, 15, 25). No post-cutoff click or assessment data is allowed into the matrix,
so an early prediction can never see late outcomes.

**Output.** One feature table per cutoff, keyed by `(code_module, code_presentation, id_student)`,
carrying the binary `at_risk` label, the four-class `final_result` (kept for the Withdrawn
in/out robustness check), and the protected attributes for the fairness and trust-equity audit.

## 0. Setup

In [16]:
from pathlib import Path

# Paths. In Colab, mount Drive and point DATA_DIR at the seven OULAD CSVs.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')

DATA_DIR = ROOT / 'data'                  # raw OULAD csv files (7 tables)
OUT_DIR  = ROOT / 'results' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment config
CUTOFF_WEEKS = [5, 10, 15, 25]            # ablation cutoffs (RQ1)
SEED = 42

PROTECTED = ['imd_band', 'disability', 'age_band', 'gender']
CONTEXT   = ['region', 'highest_education']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
RNG = np.random.default_rng(SEED)

## 1. Acquire the data

Ensures the seven OULAD CSVs are in `DATA_DIR`. It checks the folder, then searches the
rest of Drive, and only then downloads the 44.6 MB zip from the UCI mirror
(CC-BY 4.0, the same files as the Open University release). Safe to re-run: it does
nothing once the files are present.

In [18]:
import io, shutil, urllib.request, zipfile

NEEDED = ['studentInfo', 'studentRegistration', 'studentVle', 'vle',
          'studentAssessment', 'assessments', 'courses']

def have_all(d):
    return all((Path(d) / f'{n}.csv').exists() for n in NEEDED)

def extract_all_zips(folder):
    changed = True
    while changed:
        changed = False
        for z in list(Path(folder).rglob('*.zip')):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(folder)
            z.unlink(); changed = True

if have_all(DATA_DIR):
    print('All seven CSVs already in', DATA_DIR)
else:
    search_root = ROOT if ROOT.exists() else Path('/content')
    found = {n: list(search_root.rglob(f'{n}.csv')) for n in NEEDED}
    found = {n: hits[0] for n, hits in found.items() if hits}
    if len(found) == len(NEEDED):
        for n, src in found.items():
            shutil.copy(src, DATA_DIR / f'{n}.csv')
        print('Found the seven CSVs elsewhere in Drive; copied them into', DATA_DIR)
    else:
        url = 'https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip'
        print('Downloading OULAD (44.6 MB) from UCI ...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as r:
            buf = io.BytesIO(r.read())
        with zipfile.ZipFile(buf) as z:
            z.extractall(DATA_DIR)
        extract_all_zips(DATA_DIR)
        for n in NEEDED:
            if not (DATA_DIR / f'{n}.csv').exists():
                hits = list(DATA_DIR.rglob(f'{n}.csv'))
                if hits:
                    shutil.move(str(hits[0]), DATA_DIR / f'{n}.csv')

missing = [n for n in NEEDED if not (DATA_DIR / f'{n}.csv').exists()]
assert not missing, f'Still missing: {missing} - place them in {DATA_DIR} by hand.'
print('Ready:', sorted(p.name for p in DATA_DIR.glob('*.csv')))

All seven CSVs already in /content/drive/MyDrive/StudentEWS_Research/student-ews-research/data
Ready: ['assessments.csv', 'courses.csv', 'studentAssessment.csv', 'studentInfo.csv', 'studentRegistration.csv', 'studentVle.csv', 'vle.csv']


## 2. Load the seven tables

Reads each table from `DATA_DIR` (populated by the cell above).

In [19]:
def load(name):
    return pd.read_csv(DATA_DIR / f'{name}.csv')

student_info  = load('studentInfo')
registration  = load('studentRegistration')
student_vle   = load('studentVle')
vle           = load('vle')
student_assmt = load('studentAssessment')
assessments   = load('assessments')
courses       = load('courses')

for n, df in [('studentInfo', student_info), ('studentRegistration', registration),
              ('studentVle', student_vle), ('vle', vle),
              ('studentAssessment', student_assmt), ('assessments', assessments),
              ('courses', courses)]:
    print(f'{n:22s} {df.shape}')

studentInfo            (32593, 12)
studentRegistration    (32593, 5)
studentVle             (10655280, 6)
vle                    (6364, 6)
studentAssessment      (173912, 5)
assessments            (206, 6)
courses                (22, 3)


## 3. Clean numeric columns

OULAD writes some numeric fields with blanks for missing values, so pandas can read
them as strings (e.g. `score`), which breaks aggregation. Coerce them once; blanks and
stray tokens become NaN.

In [20]:
to_num = {
    'student_assmt': ['date_submitted', 'is_banked', 'score'],
    'assessments':   ['date', 'weight'],
    'student_vle':   ['date', 'sum_click'],
    'vle':           ['week_from', 'week_to'],
    'registration':  ['date_registration', 'date_unregistration'],
    'student_info':  ['num_of_prev_attempts', 'studied_credits'],
}
frames = {'student_assmt': student_assmt, 'assessments': assessments,
          'student_vle': student_vle, 'vle': vle, 'registration': registration,
          'student_info': student_info}
for nm, cols in to_num.items():
    for c in cols:
        if c in frames[nm].columns:
            frames[nm][c] = pd.to_numeric(frames[nm][c], errors='coerce')

print('score ->', student_assmt['score'].dtype, '| weight ->', assessments['weight'].dtype)

score -> float64 | weight -> float64


## 4. Keys and label

In [21]:
KEYS = ['code_module', 'code_presentation', 'id_student']

# Binary at-risk target: Fail or Withdrawn = 1, Pass or Distinction = 0.
# The original four-class result is kept for the Withdrawn in/out robustness check.
base = student_info.copy()
base['at_risk'] = base['final_result'].isin(['Fail', 'Withdrawn']).astype(int)

print(base['final_result'].value_counts())
print('\noverall at-risk rate:', round(base['at_risk'].mean(), 3))

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

overall at-risk rate: 0.528


## 5. Day to week, with the cutoff expressed in days

OULAD `date` fields are day offsets from the start of the presentation; pre-start
activity is negative. A week cutoff of `w` means we keep everything with `date <= w * 7`.

In [22]:
def to_week(day):
    return np.floor(np.asarray(day, dtype='float') / 7.0)

def cutoff_days(cutoff_week):
    return cutoff_week * 7

## 6. Cutoff-aware feature builder

For each cutoff we build three blocks of features, all clipped to the cutoff:

* **Engagement** from the VLE clickstream (volume, breadth, recency, trend, clicks by activity type).
* **Assessment** performance so far (count, mean and weighted-mean score, lateness, submission gap).
* **Registration / load** (fixed at enrolment, so no leakage risk).

`assert` statements guard against any record past the cutoff slipping in.

In [23]:
# Attach activity type to each click, once.
vle_typed = student_vle.merge(
    vle[['id_site', 'code_module', 'code_presentation', 'activity_type']],
    on=['id_site', 'code_module', 'code_presentation'], how='left')

# Attach assessment metadata (due day, type, weight) to each submission.
# studentAssessment has no module/presentation columns, so they come from `assessments`.
assmt = student_assmt.merge(
    assessments[['id_assessment', 'code_module', 'code_presentation',
                 'assessment_type', 'date', 'weight']],
    on='id_assessment', how='left').rename(columns={'date': 'due_day'})

TOP_ACTIVITIES = ['oucontent', 'quiz', 'forumng', 'resource', 'subpage', 'homepage', 'url']


def build_features(cutoff_week):
    cd = cutoff_days(cutoff_week)

    # ---- Engagement (clicks) up to cutoff ----
    v = vle_typed[vle_typed['date'] <= cd].copy()
    assert (v['date'] <= cd).all(), 'leakage: VLE record past cutoff'
    v['week'] = to_week(v['date'])

    g = v.groupby(KEYS)
    eng = pd.DataFrame({
        'clicks_total':   g['sum_click'].sum(),
        'active_days':    g['date'].nunique(),
        'active_weeks':   g['week'].nunique(),
        'distinct_sites': g['id_site'].nunique(),
    })
    eng['clicks_per_active_day'] = (eng['clicks_total'] / eng['active_days'].replace(0, np.nan))

    # Recency: clicks in the two weeks before the cutoff.
    recent = v[v['date'] > cd - 14].groupby(KEYS)['sum_click'].sum().rename('clicks_last_2w')
    eng = eng.join(recent, how='left')

    # Clicks by activity type (top types; the rest folded into 'other').
    v['atype'] = np.where(v['activity_type'].isin(TOP_ACTIVITIES), v['activity_type'], 'other')
    by_type = (v.groupby(KEYS + ['atype'])['sum_click'].sum()
                 .unstack('atype', fill_value=0).add_prefix('clicks_'))
    eng = eng.join(by_type, how='left')

    # Engagement trend: second half vs first half of the window.
    half = cd / 2.0
    early = v[v['date'] <= half].groupby(KEYS)['sum_click'].sum().rename('clicks_first_half')
    late  = v[v['date'] >  half].groupby(KEYS)['sum_click'].sum().rename('clicks_second_half')
    eng = eng.join(early, how='left').join(late, how='left')

    # ---- Assessment performance up to cutoff ----
    a = assmt[assmt['date_submitted'] <= cd].copy()
    assert (a['date_submitted'] <= cd).all(), 'leakage: submission past cutoff'
    ag = a.groupby(KEYS)
    asf = pd.DataFrame({
        'n_submitted': ag['id_assessment'].nunique(),
        'mean_score':  ag['score'].mean(),
    })
    a['wscore'] = a['score'] * a['weight']
    wsum = a.groupby(KEYS)['wscore'].sum()
    wden = a.groupby(KEYS)['weight'].sum().replace(0, np.nan)
    asf['weighted_mean_score'] = wsum / wden
    a['late'] = (a['date_submitted'] > a['due_day']).astype(float)
    asf['late_rate'] = a.groupby(KEYS)['late'].mean()

    # Submission gap: assessments due by the cutoff that were not submitted by then.
    due_set = assessments[assessments['date'] <= cd][
        ['code_module', 'code_presentation', 'id_assessment']]
    n_due = (base[KEYS].merge(due_set, on=['code_module', 'code_presentation'])
                       .groupby(KEYS)['id_assessment'].nunique().rename('n_due'))
    asf = asf.join(n_due, how='left')
    asf['n_due'] = asf['n_due'].fillna(0)
    asf['submission_gap'] = (asf['n_due'] - asf['n_submitted'].fillna(0)).clip(lower=0)

    # ---- Registration / load (set at enrolment, no leakage) ----
    reg  = registration.set_index(KEYS)[['date_registration']]
    load_cols = base.set_index(KEYS)[['num_of_prev_attempts', 'studied_credits']]

    # ---- Assemble ----
    X = (base.set_index(KEYS)[['at_risk', 'final_result'] + PROTECTED + CONTEXT]
              .join(load_cols).join(reg).join(eng).join(asf))

    # Students with no clicks / no submissions before the cutoff -> zero, not missing.
    zero_fill = ([c for c in X.columns if c.startswith('clicks_')]
                 + ['active_days', 'active_weeks', 'distinct_sites',
                    'n_submitted', 'n_due', 'submission_gap'])
    zero_fill = list(dict.fromkeys(c for c in zero_fill if c in X.columns))
    X[zero_fill] = X[zero_fill].fillna(0)

    X['cutoff_week'] = cutoff_week
    return X.reset_index()

## 7. Build and save one matrix per cutoff

In [24]:
matrices = {}
for w in CUTOFF_WEEKS:
    Xw = build_features(w)
    matrices[w] = Xw
    out = OUT_DIR / f'features_week{w}.parquet'
    try:
        Xw.to_parquet(out, index=False)
    except Exception:
        out = OUT_DIR / f'features_week{w}.csv'
        Xw.to_csv(out, index=False)
    print(f'week {w:2d}: {Xw.shape[0]:>6} rows x {Xw.shape[1]:>2} cols  ->  {out.name}')

week  5:  32593 rows x 37 cols  ->  features_week5.parquet
week 10:  32593 rows x 37 cols  ->  features_week10.parquet
week 15:  32593 rows x 37 cols  ->  features_week15.parquet
week 25:  32593 rows x 37 cols  ->  features_week25.parquet


## 8. Sanity checks: leakage, label, and first fairness look

Three things to confirm before any modelling:
1. the at-risk rate behaves sensibly across cutoffs,
2. engagement is monotonic in the cutoff (a later cutoff can only see more clicks),
3. the at-risk rate already differs by protected group, which is the whole motivation.

In [25]:
X5 = matrices[5]

print('at-risk rate by cutoff:')
for w, Xw in matrices.items():
    print(f'  week {w:2d}: {Xw["at_risk"].mean():.3f}')

print('\nat-risk rate by imd_band (week 5):')
print(X5.groupby('imd_band', dropna=False)['at_risk'].mean().round(3))

print('\nat-risk rate by disability (week 5):')
print(X5.groupby('disability', dropna=False)['at_risk'].mean().round(3))

at-risk rate by cutoff:
  week  5: 0.528
  week 10: 0.528
  week 15: 0.528
  week 25: 0.528

at-risk rate by imd_band (week 5):
imd_band
0-10%      0.648
10-20      0.614
20-30%     0.593
30-40%     0.531
40-50%     0.534
50-60%     0.512
60-70%     0.481
70-80%     0.485
80-90%     0.459
90-100%    0.425
?          0.343
Name: at_risk, dtype: float64

at-risk rate by disability (week 5):
disability
N    0.518
Y    0.619
Name: at_risk, dtype: float64


In [26]:
# Engagement must be monotonic in the cutoff: clicks_total at week 10 >= week 5 per student.
m = (matrices[5][KEYS + ['clicks_total']]
     .merge(matrices[10][KEYS + ['clicks_total']], on=KEYS, suffixes=('_w5', '_w10')))
violations = int((m['clicks_total_w10'] < m['clicks_total_w5']).sum())
print('monotonicity violations w5 -> w10:', violations, '(expected 0)')

print('\ncolumns with missing values (week 5):')
miss = X5.isna().sum()
print(miss[miss > 0] if (miss > 0).any() else 'none')

monotonicity violations w5 -> w10: 0 (expected 0)

columns with missing values (week 5):
date_registration         45
mean_score              9423
weighted_mean_score    11256
late_rate               9408
dtype: int64


In [ ]:
import shutil
from pathlib import Path

REPO = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
for sub in ['notebooks', 'utils', 'data', 'results/processed']:
    (REPO / sub).mkdir(parents=True, exist_ok=True)

NB_NAME = '01_oulad_join_features.ipynb'
target = REPO / 'notebooks' / NB_NAME

if target.exists():
    print('Already in repo:', target)
else:
    cands = list(Path('/content/drive/MyDrive').rglob(NB_NAME)) + list(Path('/content').glob(NB_NAME))
    cands = [c for c in cands if c.resolve() != target.resolve()]
    if cands:
        shutil.copy(cands[0], target)
        print('Copied', cands[0], '->', target)
    else:
        print('Could not find', NB_NAME, 'automatically.')
        print('Do File -> Save a copy in Drive, save it into', target.parent, 'then re-run.')

## Next: Notebook 02

Train the three model families (logistic regression, random forest, gradient boosting)
per cutoff with stratified splits and class-imbalance handling, then apply the hybrid
Platt/isotonic calibration and report Brier and ECE.

Decision carried into NB02: the protected attributes are kept in these matrices for the
audit, but whether they enter the models as inputs is a modelling choice. The counterfactual
step (C5) forbids perturbing `imd_band` and `disability` regardless, so mark them immutable there.